# BWOA Feature Selection on NSL-KDD

This notebook runs the Binary Whale Optimization Algorithm (BWOA) on the NSL-KDD network traffic dataset to identify the most critical and non-redundant feature subsets.

In [1]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data.nsl_kdd import NSLKDDLoader
from src.optimization.bwoa import BinaryWhaleOptimizer
from src.optimization.fitness import FeatureFitnessEvaluator
from src.utils.visualizer import ExperimentVisualizer

# Load configuration
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

nsl_train_path = config["data"]["nsl_kdd_train"]

In [2]:
# Load and preprocess data
loader = NSLKDDLoader()
train_df = loader.load(nsl_train_path)
X, y = loader.preprocess(train_df)

# Split for validation during feature selection
X_train, X_val, y_train, y_val = loader.train_test_split(X, y, test_size=0.2)
# Scale values
X_train, X_val = loader.normalize(X_train, X_val)

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")

Training features shape: (100778, 41)
Validation features shape: (25195, 41)


In [3]:
# Initialize evaluator and optimizer
bwoa_params = config["optimization"]["bwoa"]
evaluator = FeatureFitnessEvaluator(alpha=bwoa_params["alpha"])

optimizer = BinaryWhaleOptimizer(
    n_agents=bwoa_params["n_agents"],
    n_features=X_train.shape[1],
    max_iter=bwoa_params["max_iter"],
    fitness_fn=evaluator.calculate_fitness
)
print(f"BWOA configured with {optimizer.n_agents} agents and {optimizer.max_iter} iterations.")

BWOA configured with 30 agents and 100 iterations.


In [4]:
# Run optimization search loop
best_mask, fitness_history = optimizer.optimize(X_train, y_train, X_val, y_val)
print(f"Optimization complete.")
print(f"Best Fitness Score achieved: {fitness_history[-1]:.5f}")

Optimization complete.
Best Fitness Score achieved: 0.03664


In [5]:
# Plot and save fitness convergence curve
plt.figure(figsize=(8, 5))
plt.plot(fitness_history, marker="o", linestyle="-", color="b")
plt.title("BWOA Convergence Curve")
plt.xlabel("Iteration")
plt.ylabel("Fitness Value")
plt.grid(True)
plt.savefig("figures/bwoa_convergence.png")
plt.show()

<string>:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [6]:
# Print selected feature count vs original feature dimension
feature_names = loader.get_feature_names()
n_selected = int(np.sum(best_mask))
n_total = len(feature_names)
reduction = ((n_total - n_selected) / n_total) * 100.0

print(f"Original Feature Dimension: {n_total}")
print(f"BWOA Selected Feature Count: {n_selected}")
print(f"Feature Count Reduction Ratio: {reduction:.2f}%")

Original Feature Dimension: 41
BWOA Selected Feature Count: 4
Feature Count Reduction Ratio: 90.24%


In [7]:
# Plot feature selection mask
ExperimentVisualizer.plot_feature_importance(
    best_mask,
    feature_names,
    "figures/feature_importance.png"
)

# Display plot
selected_indices = np.where(best_mask == 1)[0]
selected_features = [feature_names[i] for i in selected_indices]
print(f"Selected Features: {selected_features}")

Selected Features: ['protocol_type', 'src_bytes', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate']


In [8]:
# Save selected feature mask to data/features/
np.save("data/features/nslkdd_bwoa_mask.npy", best_mask)
print("Feature selection mask successfully saved to data/features/nslkdd_bwoa_mask.npy.")

Feature selection mask successfully saved to data/features/nslkdd_bwoa_mask.npy.


### Results and Analysis

The metaheuristic whale search wrapper successfully reduced the input features dimension. Unrelated header fields and duplicated packets statistics were eliminated, which decreases classifier training computational cost while preventing overfitting in deep neural network layers.